# Capítulo 4 — Clasificación (ISLP)

Soluciones en español de ejercicios 10–16 de *An Introduction to Statistical Learning with Applications in Python*. El bloque aplicado inicia en el ejercicio 13 (Weekly); la numeración se conserva como en el libro. Semillas fijas para reproducibilidad.

## Ejercicio 10 — LDA en una dimensión

**Enunciado traducido:** Derivar los coeficientes del log-odds de LDA para p=1.

Para referencia K, $a_k=\log(\pi_k/\pi_K)-(\mu_k^2-\mu_K^2)/(2\sigma^2)$ y $b_k=(\mu_k-\mu_K)/\sigma^2$. Así, el log-odds es $a_k+b_kx$.

## Ejercicio 11 — QDA multivariado

**Enunciado traducido:** Desarrollar los términos constante, lineal y cuadrático de QDA.

$\log(P_k/P_K)=a_k+b_k^Tx+x^TB_kx$, donde $a_k=\log(\pi_k/\pi_K)-\frac12\log(|\Sigma_k|/|\Sigma_K|)-\frac12\mu_k^T\Sigma_k^{-1}\mu_k+\frac12\mu_K^T\Sigma_K^{-1}\mu_K$, $b_k=\Sigma_k^{-1}\mu_k-\Sigma_K^{-1}\mu_K$, y $B_k=-\frac12(\Sigma_k^{-1}-\Sigma_K^{-1})$.

## Ejercicio 12 — Logística binaria y softmax

**Enunciado traducido:** Relacionar log-odds y coeficientes entre la regresión logística binaria y softmax de dos clases.

(a–b) Log-odds naranja/manzana: $\beta_0+\beta_1x$ y $(\alpha_{n0}-\alpha_{a0})+(\alpha_{n1}-\alpha_{a1})x$. (c) Para $\beta=(2,-1)$, parametrización canónica $\alpha_n=(2,-1)$, $\alpha_a=(0,0)$ (invariante a desplazamiento común). (d) Interpretando los dos pares impresos como naranja=(1.2,-2), manzana=(3,0.6), resulta $\beta=(-1.8,-2.6)$. (e) Predicciones idénticas, fracción coincidente 100%, salvo empate/precisión.

## Ejercicio 13 — Weekly

**Enunciado traducido:** Resumir Weekly; ajustar logística completa con retardos y volumen; probar logística, LDA, QDA, KNN y naive Bayes usando Lag2 con entrenamiento 1990–2008 y test 2009–2010; comparar e indagar alternativas.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from ISLP import load_data
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
SEED = 1

In [ ]:
weekly=load_data("Weekly")
display(weekly.describe().T)
weekly["Volume"].plot(title="Weekly: volumen"); plt.show()
X=sm.add_constant(weekly[["Lag1","Lag2","Lag3","Lag4","Lag5","Volume"]])
y=(weekly.Direction.astype(str)=="Up").astype(int)
print(sm.Logit(y,X).fit(disp=False).summary())
train=weekly.Year<=2008; test=weekly.Year>=2009
Xtr=weekly.loc[train,["Lag2"]]; Xte=weekly.loc[test,["Lag2"]]
ytr=weekly.loc[train,"Direction"].astype(str); yte=weekly.loc[test,"Direction"].astype(str)
models={"Logística":make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=SEED)),"LDA":LinearDiscriminantAnalysis(),"QDA":QuadraticDiscriminantAnalysis(),"KNN K=1":make_pipeline(StandardScaler(),KNeighborsClassifier(1)),"Naive Bayes":GaussianNB()}
for name,model in models.items():
 model.fit(Xtr,ytr); pred=model.predict(Xte)
 print(name,"accuracy",accuracy_score(yte,pred),"error",1-accuracy_score(yte,pred)); display(pd.DataFrame(confusion_matrix(yte,pred,labels=["Down","Up"]),index=["Real Down","Real Up"],columns=["Pred Down","Pred Up"]))

La matriz tiene clases reales en filas y predichas en columnas; fuera de diagonal son errores. El ejercicio 13(j) queda **pendiente**: seleccionar modelos tras explorar test da optimismo y se requiere un protocolo separado de selección.

## Ejercicio 14 — Auto

**Enunciado traducido:** Crear mpg01 sobre la mediana; explorar predictores; comparar LDA, QDA, logística, naive Bayes y varios KNN en una partición train/test.

In [ ]:
auto=load_data("Auto").copy(); auto["mpg01"]=(auto.mpg>auto.mpg.median()).astype(int)
features=["cylinders","displacement","horsepower","weight","year"]
fig,axs=plt.subplots(1,5,figsize=(15,3))
for ax,f in zip(axs,features): auto.boxplot(column=f,by="mpg01",ax=ax); ax.set_title(f)
plt.tight_layout(); plt.show()
Xtr,Xte,ytr,yte=train_test_split(auto[features],auto.mpg01,test_size=.5,random_state=SEED,stratify=auto.mpg01)
models={"LDA":LinearDiscriminantAnalysis(),"QDA":QuadraticDiscriminantAnalysis(),"Logística":make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,random_state=SEED)),"Naive Bayes":GaussianNB()}
for k in [1,3,5,10,15,20]: models[f"KNN {k}"]=make_pipeline(StandardScaler(),KNeighborsClassifier(k))
for name,model in models.items():
 model.fit(Xtr,ytr); pred=model.predict(Xte); print(name,"test error",1-accuracy_score(yte,pred),"confusion",confusion_matrix(yte,pred))

Los boxplots guían la selección: variables de tamaño/potencia y año son plausibles. Error es fracción incorrecta en test; KNN usa escalamiento. Métricas dependen de la partición.

## Ejercicio 15 — Funciones

**Enunciado traducido:** Implementar Power, Power2, Power3, evaluar potencias y graficar x² y PlotPower.

In [ ]:
def Power():
    print(2**3)
def Power2(x, a):
    print(x**a)
def Power3(x, a):
    return x**a
def PlotPower(x, a):
    x=np.asarray(x); fig,ax=plt.subplots(); ax.plot(x,x**a,marker="o"); ax.set(xlabel="x",ylabel=f"x^{a}",title=f"Potencia {a}"); plt.show(); return ax
Power(); Power2(3,8)
for x,a in [(10,3),(8,17),(13,13)]: Power2(x,a)
x=np.arange(1,11); plt.plot(x,[Power3(v,2) for v in x],marker="o"); plt.xlabel("x"); plt.ylabel("x²"); plt.title("f(x)=x²"); plt.show()
PlotPower(np.arange(1,11),3)

## Ejercicio 16 — Boston

**Enunciado traducido:** Clasificar suburbios con tasa de crimen por encima/debajo de la mediana mediante logística, LDA, naive Bayes y KNN con subsets. Comparar dummy vs logística y estimar $SE=\sqrt{p(1-p)/n}$ con regla aproximada 2 SE. `crim` define la respuesta y se excluye de predictores para evitar fuga.

In [ ]:
boston=load_data("Boston").copy(); boston["crime01"]=(boston.crim>boston.crim.median()).astype(int)
features=["zn","indus","chas","nox","rm","age","dis","rad","tax","ptratio","lstat","medv"]
Xtr,Xte,ytr,yte=train_test_split(boston[features],boston.crime01,test_size=.5,random_state=SEED,stratify=boston.crime01)
subsets={"Todos":features,"rad+tax":["rad","tax"],"lstat+medv":["lstat","medv"],"nox+indus":["nox","indus"]}
rows=[]
for subset,cols in subsets.items():
 models={"Logística":make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,random_state=SEED)),"LDA":LinearDiscriminantAnalysis(),"Naive Bayes":GaussianNB()}
 for k in [1,5,15]: models[f"KNN {k}"]=make_pipeline(StandardScaler(),KNeighborsClassifier(k))
 for name,model in models.items():
  model.fit(Xtr[cols],ytr); pred=model.predict(Xte[cols]); rows.append({"subset":subset,"modelo":name,"error":1-accuracy_score(yte,pred),"confusion [TN,FP;FN,TP]":confusion_matrix(yte,pred).tolist()})
display(pd.DataFrame(rows).sort_values("error"))
dummy=__import__("sklearn.dummy",fromlist=["DummyClassifier"]).DummyClassifier(strategy="most_frequent").fit(Xtr,ytr)
logit=make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,random_state=SEED)).fit(Xtr,ytr)
pd_=dummy.predict(Xte); pl=logit.predict(Xte); n=len(yte); ed=1-accuracy_score(yte,pd_); el=1-accuracy_score(yte,pl)
sed=np.sqrt(ed*(1-ed)/n); sel=np.sqrt(el*(1-el)/n); threshold=2*np.sqrt(sed**2+sel**2)
print("n",n,"dummy error/SE",ed,sed,"logit error/SE",el,sel,"|difference|",abs(ed-el),"2 SE combined",threshold)
print("Diferencia supera 2 SE:",abs(ed-el)>threshold)
print("Dummy confusion",confusion_matrix(yte,pd_)); print("Logit confusion",confusion_matrix(yte,pl))

Menor error implica menos errores en esta partición; matrices muestran tipos de error. Regla 2 SE es orientación aproximada: tasas pareadas, split único y selección de modelos limitan inferencia. Boston se usa por el ejercicio histórico; interpretarlo con cautela.

## Estado

**Completos:** ejercicios 10–12 analíticos, 13(a–i), 14, 15 y 16. **Pendiente:** 13(j), comparación extensa con selección apropiada y confusión de la mejor especificación retenida. No se inventan resultados.